# VibeMatch — 01 · Chargement et découverte des données

**Projet de fin de formation GOMYCODE — Data Scientist**

VibeMatch est un système de recommandation musicale : à partir d'un morceau, je veux proposer d'autres titres qui ont la même « vibe », en me basant sur leurs caractéristiques audio.

Dans ce premier notebook, je charge les 4 fichiers du dataset Spotify (1921-2020), je regarde leur structure et je prépare une première variable dérivée, `decade`, qui me servira pour l'analyse exploratoire.

Les données se trouvent dans `../data/raw/Spotify/` (elles ne sont pas versionnées, voir le README pour les télécharger).

## 0. Imports

J'utilise **pandas** pour manipuler les tableaux et **numpy** pour les calculs numériques.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data/raw/Spotify")
pd.set_option("display.max_columns", None)

## 1. Dataset principal → `data`

`data.csv` est le fichier central du projet : **une ligne = un morceau**, avec ses caractéristiques audio (énergie, dansabilité, tempo, valence…), son année de sortie et sa popularité.

In [2]:
data = pd.read_csv(DATA_DIR / "data.csv")
data.shape

(170653, 19)

## 2. Dataset des genres → `genre_data`

`data_by_genres.csv` contient **une ligne par genre musical**, avec la moyenne des caractéristiques audio des morceaux de ce genre.

In [3]:
genre_data = pd.read_csv(DATA_DIR / "data_by_genres.csv")
genre_data.shape

(2973, 14)

## 3. Dataset des années → `year_data`

`data_by_year.csv` résume chaque **année** par la moyenne des caractéristiques de ses morceaux. Il me servira à étudier l'évolution de la musique dans le temps.

In [4]:
year_data = pd.read_csv(DATA_DIR / "data_by_year.csv")
year_data.shape

(100, 14)

## 4. Dataset des artistes → `artist_data`

`data_by_artist.csv` donne, pour chaque **artiste**, la moyenne de ses morceaux et leur nombre (`count`).

In [5]:
artist_data = pd.read_csv(DATA_DIR / "data_by_artist.csv")
artist_data.shape

(28680, 15)

## 5. Premier aperçu avec `head()`

Je regarde les 2 premières lignes de chaque DataFrame pour repérer les colonnes et le format des valeurs.

In [6]:
data.head(2)

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878,10,0.665,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.0366,80.954
1,0.9630,1921,0.732,['Dennis Day'],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000,7,0.160,-12.441,1,Clancy Lowered the Boom,5,1921,0.4150,60.936


In [7]:
genre_data.head(2)

,mode,genres,acousticness,danceability,duration_ms,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,popularity,key
0,1,21st century classical,0.979333,0.162883,1.602977e+05,0.071317,0.606834,0.3616,-31.514333,0.040567,75.336500,0.103783,27.833333,6
1,1,432hz,0.494780,0.299333,1.048887e+06,0.450678,0.477762,0.1310,-16.854000,0.076817,120.285667,0.221750,52.500000,5


In [8]:
year_data.head(2)

,mode,year,acousticness,danceability,duration_ms,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,popularity,key
0,1,1921,0.886896,0.418597,260537.166667,0.231815,0.344878,0.20571,-17.048667,0.073662,101.531493,0.379327,0.653333,2
1,1,1922,0.938592,0.482042,165469.746479,0.237815,0.434195,0.24072,-19.275282,0.116655,100.884521,0.535549,0.140845,10


In [9]:
artist_data.head(2)

,mode,count,acousticness,artists,danceability,duration_ms,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,popularity,key
0,1,9,0.590111,"""Cats"" 1981 Original London Cast",0.467222,250318.555556,0.394003,0.011400,0.290833,-14.448,0.210389,117.518111,0.389500,38.333333,5
1,1,26,0.862538,"""Cats"" 1983 Broadway Cast",0.441731,287280.000000,0.406808,0.081158,0.315215,-10.690,0.176212,103.044154,0.268865,30.576923,5


## 6. Structure de `data` et `genre_data` avec `info()`

`info()` me permet de vérifier le nombre de lignes, le type de chaque colonne et la présence de valeurs manquantes.

In [10]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 170653 entries, 0 to 170652
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   valence           170653 non-null  float64
 1   year              170653 non-null  int64  
 2   acousticness      170653 non-null  float64
 3   artists           170653 non-null  str    
 4   danceability      170653 non-null  float64
 5   duration_ms       170653 non-null  int64  
 6   energy            170653 non-null  float64
 7   explicit          170653 non-null  int64  
 8   id                170653 non-null  str    
 9   instrumentalness  170653 non-null  float64
 10  key               170653 non-null  int64  
 11  liveness          170653 non-null  float64
 12  loudness          170653 non-null  float64
 13  mode              170653 non-null  int64  
 14  name              170653 non-null  str    
 15  popularity        170653 non-null  int64  
 16  release_date      170653 non-nu

In [11]:
genre_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2973 entries, 0 to 2972
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mode              2973 non-null   int64  
 1   genres            2973 non-null   str    
 2   acousticness      2973 non-null   float64
 3   danceability      2973 non-null   float64
 4   duration_ms       2973 non-null   float64
 5   energy            2973 non-null   float64
 6   instrumentalness  2973 non-null   float64
 7   liveness          2973 non-null   float64
 8   loudness          2973 non-null   float64
 9   speechiness       2973 non-null   float64
 10  tempo             2973 non-null   float64
 11  valence           2973 non-null   float64
 12  popularity        2973 non-null   float64
 13  key               2973 non-null   int64  
dtypes: float64(11), int64(2), str(1)
memory usage: 364.8 KB


In [12]:
print("Valeurs manquantes dans data :", data.isna().sum().sum())
print("Valeurs manquantes dans genre_data :", genre_data.isna().sum().sum())
print("Doublons d'identifiant dans data :", data["id"].duplicated().sum())

Valeurs manquantes dans data : 0
Valeurs manquantes dans genre_data : 0
Doublons d'identifiant dans data : 0


**Ce que j'observe :**

- Les données sont **complètes** : aucune valeur manquante et aucun morceau en double, donc pas de nettoyage lourd à prévoir.
- La plupart des caractéristiques audio (`danceability`, `energy`, `valence`, `acousticness`…) sont déjà comprises **entre 0 et 1**, mais `tempo`, `loudness`, `duration_ms` et `popularity` sont sur des échelles très différentes. Il faudra **normaliser** avant le clustering, sinon ces colonnes domineraient les distances.
- `artists` est une liste stockée sous forme de texte (`"['Artiste A', 'Artiste B']"`) : je devrai la convertir pour pouvoir l'exploiter.
- `release_date` mélange trois formats (`AAAA-MM-JJ`, `AAAA` et `AAAA-MM`). Comme la colonne `year` est complète, je m'appuierai sur elle plutôt que sur la date.
- `data` ne contient pas de genre : l'information de genre n'existe qu'au niveau agrégé, dans `genre_data`.

## 7. Création de la colonne `decade`

Pour l'analyse, je préfère raisonner par **décennie** plutôt que par année exacte : les tendances sont plus lisibles et les graphiques moins chargés. Par exemple, 1987 devient `'1980s'`.

J'utilise `apply()` avec une fonction `lambda` : la division entière par 10, multipliée par 10, ramène chaque année au début de sa décennie.

In [13]:
data["decade"] = data["year"].apply(lambda year: f"{year // 10 * 10}s")
data[["year", "decade"]].sample(5, random_state=42)

,year,decade
116368,1977,1970s
161935,1976,1970s
135703,1994,1990s
112288,1957,1950s
22,1921,1920s


In [14]:
data["decade"].value_counts().sort_index()

decade
1920s     5126
1930s     9549
1940s    15378
1950s    19850
1960s    19549
1970s    20000
1980s    19850
1990s    19901
2000s    19646
2010s    19774
2020s     2030
Name: count, dtype: int64

De 1950 à 2010, chaque décennie compte environ 20 000 morceaux : le dataset est **plafonné à environ 2 000 titres par an**. Ces volumes ne reflètent donc pas la production musicale réelle, je ne les interpréterai pas comme une tendance. Les années 1920-1940 sont moins représentées, et la décennie 2020 ne contient que l'année 2020.

## Bilan

- 4 fichiers chargés : 170 653 morceaux, 2 973 genres, 28 680 artistes et 100 années (1921-2020).
- Données propres : aucune valeur manquante, aucun doublon.
- Points à traiter pour la suite : normalisation des variables numériques, conversion de `artists` en vraie liste.
- Nouvelle colonne `decade` prête pour l'analyse exploratoire.

**Prochaine étape :** `02_eda.ipynb`, l'analyse de l'évolution des caractéristiques audio par décennie, des genres les plus populaires et des corrélations entre variables.